# 07 · Raptor reçetesini kendi etiketlerimizle yeniden eğitmek (Kaggle, GPU)

**Neden?** Bu yarışmadaki en güçlü public model ailesi "Raptor"dur (dreaddevelopment):
CoAtNet-384 omurga + 2.5D pencereler + bulguya özgü dikkat havuzlama; tek model ~0.936–0.938.
Yazarı hem eğitim kodunu hem de önceden işlenmiş görüntü corpus'unu public paylaştı, ama
**etiketleri paylaşmadı** — eğitim script'i etiketi dışarıdan (`--labels`) bekliyor.

Leaderboard'daki 0.957 notebook'unun içindeki "R7" modeli de tam olarak bunu yapmış: aynı
reçete, kendi birleştirdikleri etiketler. Yani bu yarışmada fark **etiket kalitesinde**.
Burada aynı reçeteyi `02_etiketler` çıktımızla (`train_labels.csv`) eğitiyoruz; ileride
LLM ile iyileştirilmiş etiketler gelince sadece etiket dosyası değişecek.

| girdi | kaynak |
|---|---|
| görüntü corpus'u (4.407 vaka × 44 kesit × 336 px, ~22 GB) | `dreaddevelopment/knee-raptor-corpus` + `-ext` |
| eğitim script'i (T4 için fp16 yamalı kopya) | `rsna-knee-src` → `third_party/raptor/train_knee.py` |
| etiketler | `rsna-knee-src` → `train_labels*.csv` (sürüm aşağıda) |
| doğrulama | 58 altın vaka (eğitime girmez) |

**T4 notu:** Orijinal script bf16 kullanıyor; T4 bf16'yı donanımda desteklemediği için
yamalı kopya fp16 + GradScaler kullanır. 16 GB bellek için `bs=4` + gradient checkpointing.
`--time_budget_h` oturum sınırından önce eğitimi temiz biçimde bitirir.

**Bu sürüm (12c): Raptor-80'in üçüncü tohumu (seed 13)** — 12 ve 12b ile aynı reçete. Üç tohumun ortalaması final notebook'taki `r80` grubunda kullanılır.

In [ ]:
# Proje kökünü (src/ klasörünü içeren yer) Python yoluna ekle: yerelde ve Kaggle'da çalışır.
import os, sys
from pathlib import Path

def add_project_root():
    cands = [Path.cwd(), Path.cwd().parent]
    if Path("/kaggle/input").exists():
        # os.walk + budama: yarışmanın DICOM klasörlerine girmez (rglob dakikalar sürüyordu)
        for d, dirs, files in os.walk("/kaggle/input"):
            dirs[:] = [x for x in dirs if x not in ("train_series", "test_series")]
            if Path(d).name == "src" and "__init__.py" in files:
                cands.append(Path(d).parent)
    for c in cands:
        if (c / "src" / "__init__.py").exists():
            sys.path.insert(0, str(c))
            return c
    raise FileNotFoundError("src/ bulunamadı")

PROJECT_ROOT = add_project_root()
print("Proje kökü:", PROJECT_ROOT)

In [ ]:
import shutil, subprocess, sys
from pathlib import Path
import pandas as pd

WORK = Path("/kaggle/working")
from src.config import find_input
script = find_input("third_party/raptor/train_knee.py")
shutil.copy(script, WORK / "train_knee.py")

LABELS_FILE = "train_labels_v2.csv"
EXTRA_ARGS = ["--norm", "imagenet", "--seed", "13"]   # sürüme özgü ek argümanlar
TAG = "kadirov80_lbl2_s13"
labels_csv = find_input(LABELS_FILE)
lab = pd.read_csv(labels_csv)
lab.drop(columns=["is_gold"]).to_parquet(WORK / f"labels_{TAG}.parquet", index=False)
print(script, "|", labels_csv, "|", lab.shape)

## Eğitim

| parametre | değer | not |
|---|---|---|
| `--arch` | `coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k` | Raptor'un ana omurgası |
| `--res` | 384 | yazarın ölçümü: 256'ya düşmek −0.029 AUC |
| `--k / --k_eval` | 12 / 24 | eğitimde/değerlendirmede vaka başına pencere |
| `--epochs` | 8 | T4'te ~1–1.3 saat/epoch tahmini; bütçe aşılırsa erken durur |
| `--topk` | 3 | en iyi 3 epoch'un ağırlık ortalaması (SWA) da yazılır |

In [ ]:
cmd = [sys.executable, "train_knee.py",
       "--arch", "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k", "--res", "384",
       "--epochs", "8", "--bs", "4", "--k", "12", "--k_eval", "24", "--grad_ckpt",
       "--workers", "4", "--topk", "3", "--tag", TAG,
       "--labels", str(WORK / f"labels_{TAG}.parquet"),
       "--time_budget_h", "10.5", "--log_every", "200"] + EXTRA_ARGS
print(" ".join(cmd))
proc = subprocess.Popen(cmd, cwd=WORK, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
proc.wait()
assert proc.returncode == 0, f"eğitim hata ile bitti: {proc.returncode}"

In [ ]:
import json
res = json.load(open(WORK / f"raptor_ft_{TAG}.json"))
print("En iyi altın vaka makro AUC:", round(res["best_gold_auc"], 4), "| epoch:", res["epochs_done"])
pd.Series(res["aucs"]).round(3).to_frame("gold AUC").T